### Google colab

In [ ]:
%load_ext cuml.accel
import os
import multiprocessing

print("Nombre de cœurs disponibles :", multiprocessing.cpu_count())

In [ ]:
import os
import multiprocessing
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import warnings
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)  

In [15]:
train_df = pd.read_csv("data/train_df_FINAL2.csv")
test_df = pd.read_csv("data/test_df_FINAL3.csv")
print(len(test_df))

120526


In [16]:
## Pas obligatoire

import numpy as np

geom_cols = [
    'geometry_longitude', 'geometry_latitude', 'geometry_area', 'geometry_perimeter',
    'geometry_inscribed_circle_radius', 'geometry_compactness', 'geometry_convexity_ratio',
    'geometry_bboxwidth', 'geometry_bboxheight', 'geometry_bboxratio',
    'geometry_bboxarea', 'geometry_bboxperimeter', 'geometry_diameter',
    'geometry_vertices', 'geometry_minumum_bounding_circle', 'geometry_minimum_rotated_rectangle'
]

# Créer un masque pour toutes les lignes ayant NaN ou inf dans les colonnes géométriques
mask_invalid = np.zeros(len(train_df), dtype=bool)

for col in geom_cols:
    if col in train_df.columns:
        mask_invalid |= np.isinf(train_df[col]) | train_df[col].isna()

# Supprimer directement les lignes invalides
print(f"Lignes supprimées : {mask_invalid.sum()}")
train_df = train_df[~mask_invalid]
print(f"Taille restante : {len(train_df)}")


Lignes supprimées : 289
Taille restante : 292469


In [17]:
change_types = train_df["change_type"].unique()
for change_type in change_types:
    for column in train_df.columns:
        if column == "geometry":
            continue
        train_df.loc[(train_df["change_type"]==change_type) & (train_df[column].isnull()), column] = train_df.loc[(train_df["change_type"]==change_type), column].mode()[0]
        
for column, typ in zip(train_df.columns, train_df.dtypes):
    if typ in ['object', 'datetime64[ns]', 'geometry']:
        continue
    print('\'', end="")
    print(column, end="\', ")

'change_type', 'img_red_mean_date1', 'img_green_mean_date1', 'img_blue_mean_date1', 'img_red_std_date1', 'img_green_std_date1', 'img_blue_std_date1', 'img_red_mean_date2', 'img_green_mean_date2', 'img_blue_mean_date2', 'img_red_std_date2', 'img_green_std_date2', 'img_blue_std_date2', 'img_red_mean_date3', 'img_green_mean_date3', 'img_blue_mean_date3', 'img_red_std_date3', 'img_green_std_date3', 'img_blue_std_date3', 'img_red_mean_date4', 'img_green_mean_date4', 'img_blue_mean_date4', 'img_red_std_date4', 'img_green_std_date4', 'img_blue_std_date4', 'img_red_mean_date5', 'img_green_mean_date5', 'img_blue_mean_date5', 'img_red_std_date5', 'img_green_std_date5', 'img_blue_std_date5', 'index', 'urban_type_Dense Urban', 'urban_type_Industrial', 'urban_type_Rural', 'urban_type_Sparse Urban', 'urban_type_Urban Slum', 'urban_type_N,A', 'urban_type_Sparse Urban,Industrial', 'urban_type_Dense Urban,Industrial', 'urban_type_Sparse Urban,Urban Slum', 'urban_type_Dense Urban,Urban Slum', 'geography

In [ ]:

feature_names=[ 'urban_type_N,A', 'urban_type_Urban Slum', 'urban_type_Rural', 'urban_type_Dense Urban', 'urban_type_Industrial', 'urban_type_Sparse Urban', 
                'geography_type_N,A', 'geography_type_Desert', 'geography_type_Farms', 'geography_type_Dense Forest', 'geography_type_Hills', 'geography_type_River', 'geography_type_Grass Land', 'geography_type_Snow', 'geography_type_Lakes', 'geography_type_Barren Land', 'geography_type_Coastal', 'geography_type_Sparse Forest', 
                'img_red_mean_date0', 'img_red_mean_date1', 'img_red_mean_date2', 'img_red_mean_date3', 'img_red_mean_date4', 
                'img_green_mean_date0', 'img_green_mean_date1', 'img_green_mean_date2', 'img_green_mean_date3', 'img_green_mean_date4', 
                'img_blue_mean_date0', 'img_blue_mean_date1', 'img_blue_mean_date2', 'img_blue_mean_date3', 'img_blue_mean_date4', 
                'img_red_std_date0', 'img_red_std_date1', 'img_red_std_date2', 'img_red_std_date3', 'img_red_std_date4', 
                'img_green_std_date0', 'img_green_std_date1', 'img_green_std_date2', 'img_green_std_date3', 'img_green_std_date4', 
                'img_blue_std_date0', 'img_blue_std_date1', 'img_blue_std_date2', 'img_blue_std_date3', 'img_blue_std_date4', 
                'change_status_date0_Construction Done', 'change_status_date0_Construction Midway', 'change_status_date0_Construction Started', 'change_status_date0_Excavation', 'change_status_date0_Greenland', 'change_status_date0_Land Cleared', 'change_status_date0_Materials Dumped', 'change_status_date0_Materials Introduced', 'change_status_date0_Operational', 'change_status_date0_Prior Construction', 
                'change_status_date1_Construction Done', 'change_status_date1_Construction Midway', 'change_status_date1_Construction Started', 'change_status_date1_Excavation', 'change_status_date1_Greenland', 'change_status_date1_Land Cleared', 'change_status_date1_Materials Dumped', 'change_status_date1_Materials Introduced', 'change_status_date1_Operational', 'change_status_date1_Prior Construction', 
                'change_status_date2_Construction Done', 'change_status_date2_Construction Midway', 'change_status_date2_Construction Started', 'change_status_date2_Excavation', 'change_status_date2_Greenland', 'change_status_date2_Land Cleared', 'change_status_date2_Materials Dumped', 'change_status_date2_Materials Introduced', 'change_status_date2_Operational', 'change_status_date2_Prior Construction', 
                'change_status_date3_Construction Done', 'change_status_date3_Construction Midway', 'change_status_date3_Construction Started', 'change_status_date3_Excavation', 'change_status_date3_Greenland', 'change_status_date3_Land Cleared', 'change_status_date3_Materials Dumped', 'change_status_date3_Materials Introduced', 'change_status_date3_Operational', 'change_status_date3_Prior Construction', 
                'change_status_date4_Construction Done', 'change_status_date4_Construction Midway', 'change_status_date4_Construction Started', 'change_status_date4_Excavation', 'change_status_date4_Greenland', 'change_status_date4_Land Cleared', 'change_status_date4_Materials Dumped', 'change_status_date4_Materials Introduced', 'change_status_date4_Operational', 'change_status_date4_Prior Construction', 
                'change_status_Prior Construction', 'change_status_Greenland', 'change_status_Land Cleared', 'change_status_Excavation', 'change_status_Materials Dumped', 'change_status_Materials Introduced', 'change_status_Construction Started', 'change_status_Construction Midway', 'change_status_Construction Done', 'change_status_Operational', 
                'change_status_date0_encoded', 'change_status_date1_encoded', 'change_status_date2_encoded', 'change_status_date3_encoded', 'change_status_date4_encoded', 
                'change_status_date_4-0', 'change_status_date_1-0', 'change_status_date_2-1', 'change_status_date_3-2', 'change_status_date_4-3', 
                'img_mean_date0', 'img_std_date0', 'img_mean_date1', 'img_std_date1', 'img_mean_date2', 'img_std_date2', 'img_mean_date3', 'img_std_date3', 'img_mean_date4', 'img_std_date4', 
                'img_mean_date_4-0', 'img_mean_date_1-0', 'img_mean_date_2-1', 'img_mean_date_3-2', 'img_mean_date_4-3', 'img_std_date_4-0', 'img_std_date_1-0', 'img_std_date_2-1', 'img_std_date_3-2', 'img_std_date_4-3', 
                'img_red_mean_date_4-0', 'img_red_mean_date_1-0', 'img_red_mean_date_2-1', 'img_red_mean_date_3-2', 'img_red_mean_date_4-3', 'img_green_mean_date_4-0', 'img_green_mean_date_1-0', 'img_green_mean_date_2-1', 'img_green_mean_date_3-2', 'img_green_mean_date_4-3', 'img_blue_mean_date_4-0', 'img_blue_mean_date_1-0', 'img_blue_mean_date_2-1', 'img_blue_mean_date_3-2', 'img_blue_mean_date_4-3', 'img_red_std_date_4-0', 'img_red_std_date_1-0', 'img_red_std_date_2-1', 'img_red_std_date_3-2', 'img_red_std_date_4-3', 'img_green_std_date_4-0', 'img_green_std_date_1-0', 'img_green_std_date_2-1', 'img_green_std_date_3-2', 'img_green_std_date_4-3', 'img_blue_std_date_4-0', 'img_blue_std_date_1-0', 'img_blue_std_date_2-1', 'img_blue_std_date_3-2', 'img_blue_std_date_4-3', 
                'date_1-0', 'date_2-1', 'date_3-2', 'date_4-3', 'date_4-0', 
                'img_mean_date_4-0/Date', 'img_mean_date_1-0/Date', 'img_mean_date_2-1/Date', 'img_mean_date_3-2/Date', 'img_mean_date_4-3/Date', 'img_std_date_4-0/Date', 'img_std_date_1-0/Date', 'img_std_date_2-1/Date', 'img_std_date_3-2/Date', 'img_std_date_4-3/Date', 'img_red_mean_date_4-0/Date', 'img_red_mean_date_1-0/Date', 'img_red_mean_date_2-1/Date', 'img_red_mean_date_3-2/Date', 'img_red_mean_date_4-3/Date', 'img_green_mean_date_4-0/Date', 'img_green_mean_date_1-0/Date', 'img_green_mean_date_2-1/Date', 'img_green_mean_date_3-2/Date', 'img_green_mean_date_4-3/Date', 'img_blue_mean_date_4-0/Date', 'img_blue_mean_date_1-0/Date', 'img_blue_mean_date_2-1/Date', 'img_blue_mean_date_3-2/Date', 'img_blue_mean_date_4-3/Date', 'img_red_std_date_4-0/Date', 'img_red_std_date_1-0/Date', 'img_red_std_date_2-1/Date', 'img_red_std_date_3-2/Date', 'img_red_std_date_4-3/Date', 'img_green_std_date_4-0/Date', 'img_green_std_date_1-0/Date', 'img_green_std_date_2-1/Date', 'img_green_std_date_3-2/Date', 'img_green_std_date_4-3/Date', 'img_blue_std_date_4-0/Date', 'img_blue_std_date_1-0/Date', 'img_blue_std_date_2-1/Date', 'img_blue_std_date_3-2/Date', 'img_blue_std_date_4-3/Date', 
                'change_status_date_4-0/Date', 'change_status_date_1-0/Date', 'change_status_date_2-1/Date', 'change_status_date_3-2/Date', 'change_status_date_4-3/Date', 
                'geometry_longitude', 'geometry_latitude','geometry_area', 'geometry_perimeter', 'geometry_inscribed_circle_radius', 'geometry_compactness', 'geometry_convexity_ratio', 'geometry_vertices', 'geometry_bboxwidth', 'geometry_bboxheight', 'geometry_bboxratio', 'geometry_bboxarea', 'geometry_bboxperimeter', 
                'geometry_diameter', 'geometry_minumum_bounding_circle', 'geometry_minimum_rotated_rectangle', 'irregularity'
            ]
X = train_df[feature_names].to_numpy()
Y = train_df['change_type'].to_numpy()
X_test = test_df[feature_names].to_numpy()

# Handle NaN values using SimpleImputer
X = np.where(np.isinf(X), np.nan, X)
# imp_mean = SimpleImputer(missing_values=np.nan, strategy='median')
## Ou mask
mask_valid = ~np.isnan(X).any(axis=1)
X = X[mask_valid]
Y = Y[mask_valid]

X_train, X_valid, Y_train, Y_valid = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

In [ ]:
model = xgb.XGBClassifier(
    silent=False, 
    num_class=6,
    objective='multi:softmax', 
    seed=42,
    scale_pos_weight=1,
    learning_rate=0.03,  
    colsample_bytree=0.3,
    subsample=1.0,
    n_estimators=3000, 
    min_child_weight=29,
    max_depth=18,  # 10,14,18,15
    gamma=0.05,
    n_jobs=-1,
    reg_lambda=0.5, # L2 10?
    reg_alpha=0.05, # L1
    tree_method='hist',            #Colab : GPU : 'gpu_hist', CPU : 'hist'
    random_state=2020
)


In [ ]:
model.fit(X_train, Y_train, verbose=True)

,objective,'multi:softmax'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.3
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [ ]:
model.save_model("model_XG_D18.json")

In [ ]:
# N'est plus utilisé
pred = model.predict(X_test)  # si test_feat = X_test
sub = pd.DataFrame({"Id": np.arange(0, len(pred)), "change_type": pred})
sub.to_csv("submission_xgb_simple.csv", index=False)

In [ ]:
# Exemple proba (voir script ensemble pour tous les modèles)

proba = model.predict_proba(X_test)  

# récupérer les labels dans l'ordre du modèle
class_labels = model.classes_

# --------------------------
# 3) Build submission DataFrame
# --------------------------
PROBA_XGB = pd.DataFrame(proba, columns=[f"proba_class_{c}" for c in class_labels])
PROBA_XGB.insert(0, "Id", np.arange(len(PROBA_XGB)))

# --------------------------
# 4) Save CSV
# --------------------------
PROBA_XGB.to_csv("PROBA_XGB.csv", index=False)
print("Probabilities saved: PROBA_XGB.csv")

Probabilities saved: PROBA_XGB.csv


Validation score

In [26]:
model = xgb.XGBClassifier()
model.load_model("model/model_XG_D10_train.json")

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix


pred = model.predict(X_valid)
print(classification_report(Y_valid, pred))
print(confusion_matrix(Y_valid, pred))


              precision    recall  f1-score   support

           0       0.77      0.93      0.84      6223
           1       0.84      0.75      0.79      2829
           2       0.82      0.81      0.82     29269
           3       0.73      0.71      0.72     19882
           4       0.48      0.05      0.10       262
           5       0.00      0.00      0.00        29

    accuracy                           0.78     58494
   macro avg       0.61      0.54      0.54     58494
weighted avg       0.78      0.78      0.78     58494

[[ 5798    16   141   268     0     0]
 [   20  2127   149   533     0     0]
 [ 1208   115 23838  4098     9     1]
 [  521   268  5019 14068     6     0]
 [    9     4    58   177    14     0]
 [    0     1     5    23     0     0]]


#### Optuna pour XGboost


In [ ]:
## Très long à lancer sur GColab.
import numpy as np
import optuna
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score
from sklearn.metrics import make_scorer
from sklearn.metrics import classification_report

# ---------------- 1) Target ----------------
change_type_map = {
    'Demolition': 0,
    'Road': 1,
    'Residential': 2,
    'Commercial': 3,
    'Industrial': 4,
    'Mega Projects': 5
}

y_train_mapped = Y_train.map(change_type_map).astype(int) if Y_train.dtype == 'O' else Y_train

# ---------------- 2) Cross-validation ----------------
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

# ---------------- 3) Objective Optuna ----------------
def objective(trial):

    params = {
        "objective": "multi:softmax",
        "num_class": 6,
        "tree_method": "gpu_hist",
        "random_state": 2020,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "max_depth": trial.suggest_int("max_depth", 5, 20),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 30),
        "gamma": trial.suggest_float("gamma", 0, 0.5),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 3),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 3),
        "n_estimators": trial.suggest_int("n_estimators", 1000, 3000),
    }

    model = xgb.XGBClassifier(**params)

    scores = cross_val_score(
        model,
        X_train,
        y_train_mapped,
        cv=cv,
        scoring="f1_weighted",
        n_jobs=-1
    )

    return scores.mean()

# ---------------- 4) Study ----------------
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("Best F1 weighted:", study.best_value)
print("Best params:", study.best_params)

# ---------------- 5) Train final model ----------------
best_model = xgb.XGBClassifier(
    **study.best_params,
    objective="multi:softmax",
    num_class=6,
    tree_method="gpu_hist",
    random_state=2020,
)

best_model.fit(X_train, y_train_mapped)
best_model.save_model("xgb_model.json")  # ou .bin si tu préfères
# ---------------- 6) Evaluation ----------------
y_pred_train = best_model.predict(X_train)
print(classification_report(y_train_mapped, y_pred_train))


In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np
import geopandas as gpd
import joblib
test_df = pd.read_csv("data/test_df_FINAL3.csv")
feature_names=[ 'urban_type_N,A', 'urban_type_Urban Slum', 'urban_type_Rural', 'urban_type_Dense Urban', 'urban_type_Industrial', 'urban_type_Sparse Urban', 
                'geography_type_N,A', 'geography_type_Desert', 'geography_type_Farms', 'geography_type_Dense Forest', 'geography_type_Hills', 'geography_type_River', 'geography_type_Grass Land', 'geography_type_Snow', 'geography_type_Lakes', 'geography_type_Barren Land', 'geography_type_Coastal', 'geography_type_Sparse Forest', 
                'img_red_mean_date0', 'img_red_mean_date1', 'img_red_mean_date2', 'img_red_mean_date3', 'img_red_mean_date4', 
                'img_green_mean_date0', 'img_green_mean_date1', 'img_green_mean_date2', 'img_green_mean_date3', 'img_green_mean_date4', 
                'img_blue_mean_date0', 'img_blue_mean_date1', 'img_blue_mean_date2', 'img_blue_mean_date3', 'img_blue_mean_date4', 
                'img_red_std_date0', 'img_red_std_date1', 'img_red_std_date2', 'img_red_std_date3', 'img_red_std_date4', 
                'img_green_std_date0', 'img_green_std_date1', 'img_green_std_date2', 'img_green_std_date3', 'img_green_std_date4', 
                'img_blue_std_date0', 'img_blue_std_date1', 'img_blue_std_date2', 'img_blue_std_date3', 'img_blue_std_date4', 
                'change_status_date0_Construction Done', 'change_status_date0_Construction Midway', 'change_status_date0_Construction Started', 'change_status_date0_Excavation', 'change_status_date0_Greenland', 'change_status_date0_Land Cleared', 'change_status_date0_Materials Dumped', 'change_status_date0_Materials Introduced', 'change_status_date0_Operational', 'change_status_date0_Prior Construction', 
                'change_status_date1_Construction Done', 'change_status_date1_Construction Midway', 'change_status_date1_Construction Started', 'change_status_date1_Excavation', 'change_status_date1_Greenland', 'change_status_date1_Land Cleared', 'change_status_date1_Materials Dumped', 'change_status_date1_Materials Introduced', 'change_status_date1_Operational', 'change_status_date1_Prior Construction', 
                'change_status_date2_Construction Done', 'change_status_date2_Construction Midway', 'change_status_date2_Construction Started', 'change_status_date2_Excavation', 'change_status_date2_Greenland', 'change_status_date2_Land Cleared', 'change_status_date2_Materials Dumped', 'change_status_date2_Materials Introduced', 'change_status_date2_Operational', 'change_status_date2_Prior Construction', 
                'change_status_date3_Construction Done', 'change_status_date3_Construction Midway', 'change_status_date3_Construction Started', 'change_status_date3_Excavation', 'change_status_date3_Greenland', 'change_status_date3_Land Cleared', 'change_status_date3_Materials Dumped', 'change_status_date3_Materials Introduced', 'change_status_date3_Operational', 'change_status_date3_Prior Construction', 
                'change_status_date4_Construction Done', 'change_status_date4_Construction Midway', 'change_status_date4_Construction Started', 'change_status_date4_Excavation', 'change_status_date4_Greenland', 'change_status_date4_Land Cleared', 'change_status_date4_Materials Dumped', 'change_status_date4_Materials Introduced', 'change_status_date4_Operational', 'change_status_date4_Prior Construction', 
                'change_status_Prior Construction', 'change_status_Greenland', 'change_status_Land Cleared', 'change_status_Excavation', 'change_status_Materials Dumped', 'change_status_Materials Introduced', 'change_status_Construction Started', 'change_status_Construction Midway', 'change_status_Construction Done', 'change_status_Operational', 
                'change_status_date0_encoded', 'change_status_date1_encoded', 'change_status_date2_encoded', 'change_status_date3_encoded', 'change_status_date4_encoded', 
                'change_status_date_4-0', 'change_status_date_1-0', 'change_status_date_2-1', 'change_status_date_3-2', 'change_status_date_4-3', 
                'img_mean_date0', 'img_std_date0', 'img_mean_date1', 'img_std_date1', 'img_mean_date2', 'img_std_date2', 'img_mean_date3', 'img_std_date3', 'img_mean_date4', 'img_std_date4', 
                'img_mean_date_4-0', 'img_mean_date_1-0', 'img_mean_date_2-1', 'img_mean_date_3-2', 'img_mean_date_4-3', 'img_std_date_4-0', 'img_std_date_1-0', 'img_std_date_2-1', 'img_std_date_3-2', 'img_std_date_4-3', 
                'img_red_mean_date_4-0', 'img_red_mean_date_1-0', 'img_red_mean_date_2-1', 'img_red_mean_date_3-2', 'img_red_mean_date_4-3', 'img_green_mean_date_4-0', 'img_green_mean_date_1-0', 'img_green_mean_date_2-1', 'img_green_mean_date_3-2', 'img_green_mean_date_4-3', 'img_blue_mean_date_4-0', 'img_blue_mean_date_1-0', 'img_blue_mean_date_2-1', 'img_blue_mean_date_3-2', 'img_blue_mean_date_4-3', 'img_red_std_date_4-0', 'img_red_std_date_1-0', 'img_red_std_date_2-1', 'img_red_std_date_3-2', 'img_red_std_date_4-3', 'img_green_std_date_4-0', 'img_green_std_date_1-0', 'img_green_std_date_2-1', 'img_green_std_date_3-2', 'img_green_std_date_4-3', 'img_blue_std_date_4-0', 'img_blue_std_date_1-0', 'img_blue_std_date_2-1', 'img_blue_std_date_3-2', 'img_blue_std_date_4-3', 
                'date_1-0', 'date_2-1', 'date_3-2', 'date_4-3', 'date_4-0', 
                'img_mean_date_4-0/Date', 'img_mean_date_1-0/Date', 'img_mean_date_2-1/Date', 'img_mean_date_3-2/Date', 'img_mean_date_4-3/Date', 'img_std_date_4-0/Date', 'img_std_date_1-0/Date', 'img_std_date_2-1/Date', 'img_std_date_3-2/Date', 'img_std_date_4-3/Date', 'img_red_mean_date_4-0/Date', 'img_red_mean_date_1-0/Date', 'img_red_mean_date_2-1/Date', 'img_red_mean_date_3-2/Date', 'img_red_mean_date_4-3/Date', 'img_green_mean_date_4-0/Date', 'img_green_mean_date_1-0/Date', 'img_green_mean_date_2-1/Date', 'img_green_mean_date_3-2/Date', 'img_green_mean_date_4-3/Date', 'img_blue_mean_date_4-0/Date', 'img_blue_mean_date_1-0/Date', 'img_blue_mean_date_2-1/Date', 'img_blue_mean_date_3-2/Date', 'img_blue_mean_date_4-3/Date', 'img_red_std_date_4-0/Date', 'img_red_std_date_1-0/Date', 'img_red_std_date_2-1/Date', 'img_red_std_date_3-2/Date', 'img_red_std_date_4-3/Date', 'img_green_std_date_4-0/Date', 'img_green_std_date_1-0/Date', 'img_green_std_date_2-1/Date', 'img_green_std_date_3-2/Date', 'img_green_std_date_4-3/Date', 'img_blue_std_date_4-0/Date', 'img_blue_std_date_1-0/Date', 'img_blue_std_date_2-1/Date', 'img_blue_std_date_3-2/Date', 'img_blue_std_date_4-3/Date', 
                'change_status_date_4-0/Date', 'change_status_date_1-0/Date', 'change_status_date_2-1/Date', 'change_status_date_3-2/Date', 'change_status_date_4-3/Date', 
                'geometry_longitude', 'geometry_latitude','geometry_area', 'geometry_perimeter', 'geometry_inscribed_circle_radius', 'geometry_compactness', 'geometry_convexity_ratio', 'geometry_vertices', 'geometry_bboxwidth', 'geometry_bboxheight', 'geometry_bboxratio', 'geometry_bboxarea', 'geometry_bboxperimeter', 
                'geometry_diameter', 'geometry_minumum_bounding_circle', 'geometry_minimum_rotated_rectangle', 'irregularity', 
            ]
X_test = test_df[feature_names].to_numpy()

# Recréer un modèle vide
model = xgb.XGBClassifier()

# Load les poids
model.load_model("model/model_XG_D15.json")

# Prédictions directes (labels)
pred_labels = model.predict(X_test)

# Submission DataFrame
PRED_XGB = pd.DataFrame({
    "Id": np.arange(len(pred_labels)),
    "change_type": pred_labels  # Ici on remplace par les labels prédits
})

# Save CSV
PRED_XGB.to_csv("submission_solo/sample_submission_XG15.csv", index=False)
print("Predictions saved: sample_submission_XG15.csv")


#### Catboost

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import numpy as np

# =========================
# DATA
# =========================
X = train_df[feature_names].to_numpy()
Y = train_df['change_type'].to_numpy()
X_test = test_df[feature_names].to_numpy()

# Nettoyage NaN / inf
X = np.where(np.isinf(X), np.nan, X)
mask_valid = ~np.isnan(X).any(axis=1)
X = X[mask_valid]
Y = Y[mask_valid]

X_train, X_valid, Y_train, Y_valid = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=65,
    stratify=Y
)

print(np.unique(Y_train, return_counts=True))

# =========================
# METRICS
# =========================
def printScores(y_true, y_predict):
    print("F1 score micro:", f1_score(y_true, y_predict, average='micro'))
    print("F1 score macro:", f1_score(y_true, y_predict, average='macro'))
    print("F1 score weighted:", f1_score(y_true, y_predict, average='weighted'))
    print(confusion_matrix(y_true, y_predict))


# =========================
# CATBOOST MODEL
# =========================
model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',        # Très bon pour classes déséquilibrées
    iterations=4000,
    learning_rate=0.03,
    depth=10,
    l2_leaf_reg=6,                # équivalent reg_lambda
    random_strength=1.0,
    bagging_temperature=1.0,
    border_count=128,
    task_type='GPU',              # GPU
    devices='0',
    random_seed=2020,
    verbose=200
)

# =========================
# TRAIN
# =========================
model.fit(
    X_train, Y_train,
    eval_set=(X_valid, Y_valid),
    use_best_model=True
)

# =========================
# EVAL
# =========================
pred = model.predict(X_valid)
pred = pred.squeeze()  # CatBoost retourne (n,1)

printScores(Y_valid, pred)

# =========================
# SAVE
# =========================
# Fit sur tout le train

model.save_model("model_CATBOOST_D10.cbm")


In [ ]:
#best_iter = 3291  # valeur obtenue avec use_best_model=True D12
best_iter = 3800  # valeur obtenue avec use_best_model=True D10

# Nouveau modèle final
final_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    iterations=best_iter,   # on s'arrête exactement là où le modèle était "best"
    learning_rate=0.03,
    depth=10,
    l2_leaf_reg=6,
    random_strength=1.0,
    bagging_temperature=1.0,
    border_count=128,
    task_type='GPU',
    devices='0',
    random_seed=2020,
    verbose=200
)

# D12 : L2 = 5 sinon identique

final_model.fit(
    X, Y,
    use_best_model=False  # on a déjà limité iterations
)


model.save_model("model_CATBOOST_D10.cbm")

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier


## PREPROCCESSING XG
import numpy as np
import pandas as pd
import geopandas as gpd
import joblib
test_df = pd.read_csv("data/test_df_FINAL3.csv")
feature_names=[ 'urban_type_N,A', 'urban_type_Urban Slum', 'urban_type_Rural', 'urban_type_Dense Urban', 'urban_type_Industrial', 'urban_type_Sparse Urban', 
                'geography_type_N,A', 'geography_type_Desert', 'geography_type_Farms', 'geography_type_Dense Forest', 'geography_type_Hills', 'geography_type_River', 'geography_type_Grass Land', 'geography_type_Snow', 'geography_type_Lakes', 'geography_type_Barren Land', 'geography_type_Coastal', 'geography_type_Sparse Forest', 
                'img_red_mean_date0', 'img_red_mean_date1', 'img_red_mean_date2', 'img_red_mean_date3', 'img_red_mean_date4', 
                'img_green_mean_date0', 'img_green_mean_date1', 'img_green_mean_date2', 'img_green_mean_date3', 'img_green_mean_date4', 
                'img_blue_mean_date0', 'img_blue_mean_date1', 'img_blue_mean_date2', 'img_blue_mean_date3', 'img_blue_mean_date4', 
                'img_red_std_date0', 'img_red_std_date1', 'img_red_std_date2', 'img_red_std_date3', 'img_red_std_date4', 
                'img_green_std_date0', 'img_green_std_date1', 'img_green_std_date2', 'img_green_std_date3', 'img_green_std_date4', 
                'img_blue_std_date0', 'img_blue_std_date1', 'img_blue_std_date2', 'img_blue_std_date3', 'img_blue_std_date4', 
                'change_status_date0_Construction Done', 'change_status_date0_Construction Midway', 'change_status_date0_Construction Started', 'change_status_date0_Excavation', 'change_status_date0_Greenland', 'change_status_date0_Land Cleared', 'change_status_date0_Materials Dumped', 'change_status_date0_Materials Introduced', 'change_status_date0_Operational', 'change_status_date0_Prior Construction', 
                'change_status_date1_Construction Done', 'change_status_date1_Construction Midway', 'change_status_date1_Construction Started', 'change_status_date1_Excavation', 'change_status_date1_Greenland', 'change_status_date1_Land Cleared', 'change_status_date1_Materials Dumped', 'change_status_date1_Materials Introduced', 'change_status_date1_Operational', 'change_status_date1_Prior Construction', 
                'change_status_date2_Construction Done', 'change_status_date2_Construction Midway', 'change_status_date2_Construction Started', 'change_status_date2_Excavation', 'change_status_date2_Greenland', 'change_status_date2_Land Cleared', 'change_status_date2_Materials Dumped', 'change_status_date2_Materials Introduced', 'change_status_date2_Operational', 'change_status_date2_Prior Construction', 
                'change_status_date3_Construction Done', 'change_status_date3_Construction Midway', 'change_status_date3_Construction Started', 'change_status_date3_Excavation', 'change_status_date3_Greenland', 'change_status_date3_Land Cleared', 'change_status_date3_Materials Dumped', 'change_status_date3_Materials Introduced', 'change_status_date3_Operational', 'change_status_date3_Prior Construction', 
                'change_status_date4_Construction Done', 'change_status_date4_Construction Midway', 'change_status_date4_Construction Started', 'change_status_date4_Excavation', 'change_status_date4_Greenland', 'change_status_date4_Land Cleared', 'change_status_date4_Materials Dumped', 'change_status_date4_Materials Introduced', 'change_status_date4_Operational', 'change_status_date4_Prior Construction', 
                'change_status_Prior Construction', 'change_status_Greenland', 'change_status_Land Cleared', 'change_status_Excavation', 'change_status_Materials Dumped', 'change_status_Materials Introduced', 'change_status_Construction Started', 'change_status_Construction Midway', 'change_status_Construction Done', 'change_status_Operational', 
                'change_status_date0_encoded', 'change_status_date1_encoded', 'change_status_date2_encoded', 'change_status_date3_encoded', 'change_status_date4_encoded', 
                'change_status_date_4-0', 'change_status_date_1-0', 'change_status_date_2-1', 'change_status_date_3-2', 'change_status_date_4-3', 
                'img_mean_date0', 'img_std_date0', 'img_mean_date1', 'img_std_date1', 'img_mean_date2', 'img_std_date2', 'img_mean_date3', 'img_std_date3', 'img_mean_date4', 'img_std_date4', 
                'img_mean_date_4-0', 'img_mean_date_1-0', 'img_mean_date_2-1', 'img_mean_date_3-2', 'img_mean_date_4-3', 'img_std_date_4-0', 'img_std_date_1-0', 'img_std_date_2-1', 'img_std_date_3-2', 'img_std_date_4-3', 
                'img_red_mean_date_4-0', 'img_red_mean_date_1-0', 'img_red_mean_date_2-1', 'img_red_mean_date_3-2', 'img_red_mean_date_4-3', 'img_green_mean_date_4-0', 'img_green_mean_date_1-0', 'img_green_mean_date_2-1', 'img_green_mean_date_3-2', 'img_green_mean_date_4-3', 'img_blue_mean_date_4-0', 'img_blue_mean_date_1-0', 'img_blue_mean_date_2-1', 'img_blue_mean_date_3-2', 'img_blue_mean_date_4-3', 'img_red_std_date_4-0', 'img_red_std_date_1-0', 'img_red_std_date_2-1', 'img_red_std_date_3-2', 'img_red_std_date_4-3', 'img_green_std_date_4-0', 'img_green_std_date_1-0', 'img_green_std_date_2-1', 'img_green_std_date_3-2', 'img_green_std_date_4-3', 'img_blue_std_date_4-0', 'img_blue_std_date_1-0', 'img_blue_std_date_2-1', 'img_blue_std_date_3-2', 'img_blue_std_date_4-3', 
                'date_1-0', 'date_2-1', 'date_3-2', 'date_4-3', 'date_4-0', 
                'img_mean_date_4-0/Date', 'img_mean_date_1-0/Date', 'img_mean_date_2-1/Date', 'img_mean_date_3-2/Date', 'img_mean_date_4-3/Date', 'img_std_date_4-0/Date', 'img_std_date_1-0/Date', 'img_std_date_2-1/Date', 'img_std_date_3-2/Date', 'img_std_date_4-3/Date', 'img_red_mean_date_4-0/Date', 'img_red_mean_date_1-0/Date', 'img_red_mean_date_2-1/Date', 'img_red_mean_date_3-2/Date', 'img_red_mean_date_4-3/Date', 'img_green_mean_date_4-0/Date', 'img_green_mean_date_1-0/Date', 'img_green_mean_date_2-1/Date', 'img_green_mean_date_3-2/Date', 'img_green_mean_date_4-3/Date', 'img_blue_mean_date_4-0/Date', 'img_blue_mean_date_1-0/Date', 'img_blue_mean_date_2-1/Date', 'img_blue_mean_date_3-2/Date', 'img_blue_mean_date_4-3/Date', 'img_red_std_date_4-0/Date', 'img_red_std_date_1-0/Date', 'img_red_std_date_2-1/Date', 'img_red_std_date_3-2/Date', 'img_red_std_date_4-3/Date', 'img_green_std_date_4-0/Date', 'img_green_std_date_1-0/Date', 'img_green_std_date_2-1/Date', 'img_green_std_date_3-2/Date', 'img_green_std_date_4-3/Date', 'img_blue_std_date_4-0/Date', 'img_blue_std_date_1-0/Date', 'img_blue_std_date_2-1/Date', 'img_blue_std_date_3-2/Date', 'img_blue_std_date_4-3/Date', 
                'change_status_date_4-0/Date', 'change_status_date_1-0/Date', 'change_status_date_2-1/Date', 'change_status_date_3-2/Date', 'change_status_date_4-3/Date', 
                'geometry_longitude', 'geometry_latitude','geometry_area', 'geometry_perimeter', 'geometry_inscribed_circle_radius', 'geometry_compactness', 'geometry_convexity_ratio', 'geometry_vertices', 'geometry_bboxwidth', 'geometry_bboxheight', 'geometry_bboxratio', 'geometry_bboxarea', 'geometry_bboxperimeter', 
                'geometry_diameter', 'geometry_minumum_bounding_circle', 'geometry_minimum_rotated_rectangle', 'irregularity', 
            ]
X_test = test_df[feature_names].to_numpy()

## CatBoost D_10

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier


# Load trained CatBoost model
model = CatBoostClassifier()
model.load_model("model/model_CATBOOST_D10.cbm")

# Predict class labels directly
pred_labels = model.predict(X_test).ravel()  # flatten to 1D

# Submission DataFrame
SUBMISSION_CAT = pd.DataFrame({
    "Id": np.arange(len(pred_labels)),
    "change_type": pred_labels
})

# Save CSV
SUBMISSION_CAT.to_csv("submission_solo/SUBMISSION_CATBOOST_D10.csv", index=False)
print("Predictions saved: SUBMISSION_CATBOOST_D10.csv")

